## tl;dr
10개 선정 모델의 검증 20건을 반복한 200개 예측에서 80개 오류, 모든 실행에서 틀린 사례 4개다. 기능 설명 누락을 확인하고 합성 train을 60→240으로 늘렸다. 인과·실사용 개선은 아직 확인하지 않았다.

## Context & Methods
검증 예측만 사용한다. Test 예측은 읽지 않는다.
### Key Assumptions
라벨은 미검토이며 카탈로그 대조는 실행 성공의 증거가 아니다. Python으로 셀을 순서 실행한 출력이다. Jupyter kernel 실행은 nbformat/nbclient/ipykernel 미설치로 수행하지 않았다.

In [1]:
from pathlib import Path
import json, sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/audit_ten_tool_errors.py').exists())
sys.path.insert(0, str(root / 'scripts'))
from audit_ten_tool_errors import audit


## Data
원본 full validation export와 선택 head 예측 10세트. 예측 원문은 outputs에 보관하고 MLflow 감사 artifact로 제공한다.

In [2]:
validation = root / 'outputs/laya-ten-tools-20261020/full/validation.jsonl'
predictions = root / 'outputs/laya-ten-tools-error-review-20261020/validation-error-input.json'
result = audit(validation, predictions)
print({k: result[k] for k in ('runs','cases','prediction_events','wrong_events','cases_wrong_in_any_run','cases_wrong_in_every_run')})


{'runs': 10, 'cases': 20, 'prediction_events': 200, 'wrong_events': 80, 'cases_wrong_in_any_run': 10, 'cases_wrong_in_every_run': 4}


## Results
### 반복 혼동 상위 5개

In [3]:
for pair in result['confusions'][:5]:
    print(pair)


{'gold': 'mcp__obsidian_vault__bases', 'predicted': 'mcp__obsidian_vault__dataview', 'count': 10}
{'gold': 'mcp__swallow_trader__swallow_get_price_history', 'predicted': 'mcp__swallow_trader__swallow_get_stock_price', 'count': 10}
{'gold': 'mcp__swallow_trader__swallow_get_stock_price', 'predicted': 'mcp__swallow_trader__swallow_get_price_history', 'count': 9}
{'gold': 'mcp__swallow_trader__swallow_get_portfolio', 'predicted': 'mcp__swallow_trader__swallow_order', 'count': 8}
{'gold': 'mcp__swallow_trader__swallow_get_portfolio', 'predicted': 'mcp__swallow_trader__swallow_get_stock_price', 'count': 7}


### 새 profile과 held-out 분리

In [4]:
manifest = json.loads((root / 'datasets/hermes-ten-tools-v2/manifest.json').read_text())
for name, splits in manifest['profiles'].items():
    print(name, {split: stats['rows'] for split, stats in splits.items()})
for split in ('validation','test'):
    assert manifest['profiles']['description_control'][split]['sha256'] == manifest['profiles']['augmented'][split]['sha256']
print('Both controls share identical validation/test exports; source states/labels retained by builder tests.')


description_control {'train': 60, 'validation': 20, 'test': 20}
augmented {'train': 240, 'validation': 20, 'test': 20}
Both controls share identical validation/test exports; source states/labels retained by builder tests.


## Takeaways
기능 설명 보완과 학습 데이터 보강을 분리한 회귀 실험을 준비했다. 신규 자료도 AI 작성·미검토다. 독립적인 실제 요청 평가와 사람 정답 검토가 필요하다.

Jupyter 실행 재현(별도 notebook 환경): `python -m jupyter nbconvert --execute --to notebook --inplace docs/data/ten-tool-error-review-20261020.ipynb`